# Part2: Cell subtype identification of malignant cells in the HGSC dataset (Continued from previous code)

# Package loading and path setting

In [5]:
import os
import numpy as np
import pandas as pd
import scanpy as sc
import torch
from torch_scatter import scatter_max, scatter_add
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from scipy.stats import spearmanr


In [6]:
def scatter_nanmax(src: torch.Tensor, index: torch.Tensor, dim: int = 0, dim_size: int = None) -> torch.Tensor:
    """
    Similar to scatter_max, but ignores NaN values when computing the group-wise maximum.
    If all elements in a group are NaN, output is set to NaN for that group.
    """
    mask = (~torch.isnan(src)).float()
    src_no_nan = torch.nan_to_num(src, nan=float("-inf"))
    out, _ = scatter_max(src_no_nan, index, dim=dim, dim_size=dim_size)
    den = scatter_add(mask, index, dim=dim, dim_size=dim_size)
    out[den == 0] = float("nan")
    return out

# Paths and devices

In [7]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

file_savepath_main = r'D:/CellFlowMap/HGSC/Results/V1/SpiderNet_Result_Mode_cell_class_selfsetdim15/'
data_path_main = r'D:/CellFlowMap/HGSC/Data/'

Using device: cuda


# load data

In [8]:
# Load data
Factor_envir_list = pd.read_pickle(file_savepath_main + "Factor_envir_list.pkl")
SpiderNet_data_pyg_list = pd.read_pickle(file_savepath_main + 'SpiderNet_data_pyg_list.pkl')
adata_copy = sc.read_h5ad(file_savepath_main + "adata_all.h5ad")
adata_choose = sc.read_h5ad(file_savepath_main + "adata_choose_Malignant.h5ad")
adata_list = pd.read_pickle(file_savepath_main + 'adata_list.pkl')

cellclass_unique = np.unique(adata_copy.obs['cell.types'])
if np.min(adata_choose.obs['MI_louvain'].astype(int)) == 0:
    adata_choose.obs['MI_louvain'] = (adata_choose.obs['MI_louvain'].astype(int) + 1).astype(str)

match_index = adata_copy.obs.index.get_indexer(adata_choose.obs['barcode'])
barcode_all = adata_copy.obs.index.tolist()

# Update malignant cluster annotation
malignant_cluster = ['Malignant_' + str(c) for c in adata_choose.obs['MI_louvain']]
malignant_map = dict(zip(adata_choose.obs['barcode'], malignant_cluster))
cellclass = adata_copy.obs['cell.types'].values
cellclass_updated = np.array([malignant_map.get(b, c) for b, c in zip(barcode_all, cellclass)])
cellclass_updated_df = pd.DataFrame({'barcode': barcode_all, 'cell.types.updated': cellclass_updated}).set_index('barcode')

# Step 1. Compute MI sender/receiver aggregation per cell

In [9]:
MI_SR_agg_cur_all, MI_SR_agg_ct_all = [], []

for slice_index, Factor_envir in enumerate(Factor_envir_list):
    Factor_envir = torch.tensor(Factor_envir, dtype=torch.float32, device=device)
    edge_index = SpiderNet_data_pyg_list[slice_index]['edge_index']
    num_cell = SpiderNet_data_pyg_list[slice_index].x.shape[0]
    cell_class = SpiderNet_data_pyg_list[slice_index]['cell_class']
    barcodes = SpiderNet_data_pyg_list[slice_index]['cellnames']
    cellclass_updated_cur = cellclass_updated_df.loc[barcodes, 'cell.types.updated'].values
    sender_class = cell_class[edge_index[:, 0].cpu().numpy()]
    receiver_class = cell_class[edge_index[:, 1].cpu().numpy()]

    MI_SR_agg_cur_sub_list = []
    for ctype in cellclass_unique:
        send_idx = np.where(sender_class == ctype)[0]
        recv_idx = np.where(receiver_class == ctype)[0]

        F_recv = np.full(Factor_envir.shape, np.nan, dtype=np.float32)
        F_send = np.full(Factor_envir.shape, np.nan, dtype=np.float32)
        F_recv[recv_idx, :] = Factor_envir[recv_idx, :].cpu().numpy()
        F_send[send_idx, :] = Factor_envir[send_idx, :].cpu().numpy()

        MI_recv = scatter_nanmax(torch.tensor(F_send, device=device), edge_index[:, 1].long().to(device),
                                 dim=0, dim_size=num_cell).cpu().numpy()
        MI_send = scatter_nanmax(torch.tensor(F_recv, device=device), edge_index[:, 0].long().to(device),
                                 dim=0, dim_size=num_cell).cpu().numpy()

        MI_SR_agg_cur_sub_list.append(np.hstack([MI_send, MI_recv]))

    MI_SR_agg_ct = np.hstack(MI_SR_agg_cur_sub_list)
    MI_SR_agg_ct_all.append(MI_SR_agg_ct)

    MI_recv_all, _ = scatter_max(Factor_envir, edge_index[:, 1].long().to(device), dim=0, dim_size=num_cell)
    MI_send_all, _ = scatter_max(Factor_envir, edge_index[:, 0].long().to(device), dim=0, dim_size=num_cell)
    MI_SR_agg_cur_all.append(np.hstack([MI_send_all.cpu().numpy(), MI_recv_all.cpu().numpy()]))

    if slice_index == 0:
        MI_agg_meta = pd.DataFrame({
            "MI": [f"MI{i+1}" for i in range(MI_send_all.shape[1])] * 2,
            "SR": ["Sender"] * MI_send_all.shape[1] + ["Receiver"] * MI_recv_all.shape[1]
        })
        MI_agg_ct_meta = pd.concat([
            MI_agg_meta.assign(celltype=ctype) for ctype in cellclass_unique
        ], ignore_index=True)

MI_SR_agg_cur_all = np.vstack(MI_SR_agg_cur_all)
MI_SR_agg_ct_all = np.vstack(MI_SR_agg_ct_all)

# Step 2. Cluster-level MI aggregation and visualization

## MI sending/receiving Levels per Cluster

In [14]:
# Prepare data and assign proper column names
MI_SR_agg_cur_all_choose = MI_SR_agg_cur_all[match_index, :]
cluster_label = adata_choose.obs["MI_louvain"].astype(str).values

MI_column_names = [f"{row.MI}_{row.SR}" for _, row in MI_agg_meta.iterrows()]
df_MI = pd.DataFrame(
    MI_SR_agg_cur_all_choose,
    index=adata_choose.obs["barcode"],
    columns=MI_column_names
)
df_MI["cluster"] = cluster_label

# Compute mean per cluster
cluster_mean = df_MI.groupby("cluster").mean()
cluster_mean.index = [f"Cluster_{i}" for i in cluster_mean.index]

# --------------------------------------------------------------
# Reorder clusters and split sender/receiver MIs
# --------------------------------------------------------------
cluster_order = ["Cluster_4", "Cluster_2", "Cluster_6", "Cluster_1", "Cluster_5", "Cluster_3"]
cluster_mean_order = cluster_mean.loc[[c for c in cluster_order if c in cluster_mean.index], :]

cluster_mean_order_sender = cluster_mean_order.loc[:, [col for col in cluster_mean_order.columns if "Sender" in col]]
cluster_mean_order_receiver = cluster_mean_order.loc[:, [col for col in cluster_mean_order.columns if "Receiver" in col]]

# ==============================================================
# Heatmap 1: MI Sending Levels per Cluster
# ==============================================================
from matplotlib.colors import LinearSegmentedColormap

cmap_sender = LinearSegmentedColormap.from_list("white_to_green", ["#FFFFFF", "#A8DADC"])

plt.figure(figsize=(12, 6))
sns.heatmap(
    cluster_mean_order_sender,
    cmap=cmap_sender,
    annot=True,
    fmt=".2f",
    vmax=0.4,
    vmin=0,
    cbar_kws={'label': 'Mean MI Sending Level'}
)
plt.title("Mean MI Sending Levels per Cluster")
plt.xlabel("MI Factors")
plt.ylabel("Clusters")
plt.tight_layout()
plt.savefig(
    file_savepath_main + "MI_Sending_Levels_per_Cluster.png",
    dpi=300,
    bbox_inches='tight',
    facecolor="white"
)
plt.close()

# ==============================================================
# Heatmap 2: MI Receiving Levels per Cluster
# ==============================================================
cmap_receiver = LinearSegmentedColormap.from_list("white_to_orange", ["#FFFFFF", "#FDBE85"])

plt.figure(figsize=(12, 6))
sns.heatmap(
    cluster_mean_order_receiver,
    cmap=cmap_receiver,
    annot=True,
    fmt=".2f",
    vmax=0.4,
    vmin=0,
    cbar_kws={'label': 'Mean MI Receiving Level'}
)
plt.title("Mean MI Receiving Levels per Cluster")
plt.xlabel("MI Factors")
plt.ylabel("Clusters")
plt.tight_layout()
plt.savefig(
    file_savepath_main + "MI_Receiving_Levels_per_Cluster.png",
    dpi=300,
    bbox_inches='tight',
    facecolor="white"
)
plt.close()



✅ Saved ordered sender and receiver MI heatmaps.


# Mean MI sending/receiving levels between malignant clusters and cell types

In [15]:
# Define MI–direction pairs
MI_OI_df = pd.DataFrame({
    "MI": ['MI10', 'MI13', 'MI6', 'MI9', 'MI12'],
    "direction": ['Receiver'] * 5
})

# Custom colormap: white → green
cmap_custom = LinearSegmentedColormap.from_list("white_to_green", ["#FFFFFF", "#CCEAA8", "#0C6338"])

# Cluster display order (skip missing ones automatically)
cluster_order = ["Cluster_4", "Cluster_2", "Cluster_6", "Cluster_3", "Cluster_5", "Cluster_1"]

# Create subplot figure
fig, axes = plt.subplots(1, MI_OI_df.shape[0], figsize=(6 * MI_OI_df.shape[0], 6), sharey=True)

for i, (MI_OI, direction_OI) in enumerate(zip(MI_OI_df['MI'], MI_OI_df['direction'])):
    # Find indices for current MI and direction
    MI_ct_index = MI_agg_ct_meta[
        (MI_agg_ct_meta['MI'] == MI_OI) & (MI_agg_ct_meta['SR'] == direction_OI)
    ].index

    if len(MI_ct_index) == 0:
        axes[i].axis('off')
        axes[i].set_title(f"No data for {MI_OI} {direction_OI}", fontsize=12)
        continue

    MI_meta_OI = MI_agg_ct_meta.loc[MI_ct_index, :].copy()
    MI_values = MI_SR_agg_ct_all[match_index, :][:, MI_ct_index]

    df_MI_ct = pd.DataFrame(MI_values, index=adata_choose.obs['barcode'])
    df_MI_ct['cluster'] = cluster_label

    # Compute cluster-wise mean
    cluster_ct_mean = df_MI_ct.groupby('cluster').mean(numeric_only=True)
    cluster_ct_mean.index = [f'Cluster_{idx}' for idx in cluster_ct_mean.index]

    # Assign cell types as column names
    if len(MI_meta_OI['celltype'].values) == cluster_ct_mean.shape[1]:
        cluster_ct_mean.columns = MI_meta_OI['celltype'].values
    else:
        cluster_ct_mean.columns = [f"celltype_{j+1}" for j in range(cluster_ct_mean.shape[1])]

    # Reorder clusters
    cluster_ct_mean = cluster_ct_mean.loc[[c for c in cluster_order if c in cluster_ct_mean.index], :]

    # Plot heatmap
    sns.heatmap(
        cluster_ct_mean.astype(float),
        cmap=cmap_custom,
        annot=False,
        fmt=".2f",
        vmax=0.7,
        vmin=0.0,
        ax=axes[i],
        cbar=(i == MI_OI_df.shape[0] - 1),
        cbar_kws={'label': f"Mean {MI_OI} {direction_OI} Level"}
    )

    # Titles and labels
    if direction_OI.lower() == "receiver":
        title = f"Receiving {MI_OI}"
        xlabel = "Sender Cell Type"
    else:
        title = f"Sending {MI_OI}"
        xlabel = "Receiver Cell Type"

    axes[i].set_title(title, fontsize=14)
    axes[i].set_xlabel(xlabel, fontsize=12)
    axes[i].set_ylabel("Clusters" if i == 0 else "")

# Save figure
plt.tight_layout()
plt.savefig(
    file_savepath_main + "Combined_MI_Levels_per_Cluster_CellType.png",
    dpi=300,
    bbox_inches="tight",
    facecolor="white"
)
plt.close()


# Step 3. Correlation with CAF score

In [11]:
exp_nor_all = adata_copy.X.toarray()
exp_nor_all_norm = exp_nor_all / (np.max(exp_nor_all, axis=0) + 1e-6)

caf_modules = {
    "myCAF": ["ACTA2","TAGLN","MYL9","TPM2","CNN1","CALD1","COL1A1","COL1A2"],
    "iCAF": ["IL6","CXCL12","CXCL14","LIF","CCL2","PTGS2"],
    "apCAF": ["HLA-DRA","HLA-DRB1","CD74","CIITA"],
    "meCAF": ["COL11A1","THBS2","MMP11","ITGA11","FN1","VCAN","SPARC","SULF1","LOX","PLOD2"],
    "periCAF": ["RGS5","PDGFRB","MCAM","NOTCH3","TAGLN"],
    "prolCAF": ["MKI67","TOP2A","PCNA"]
}

caf_genes = np.unique([g for genes in caf_modules.values() for g in genes if g in adata_copy.var_names])
caf_score = np.mean(exp_nor_all_norm[:, np.isin(adata_copy.var_names, caf_genes)], axis=1)
fibro_idx = np.where(cellclass_updated == 'Fibroblast')[0]

MI_OI_df = pd.DataFrame({
    "MI": ['MI10','MI13','MI6','MI9','MI12'],
    "direction": ['Sender']*5
})

corr_CAF_df = pd.DataFrame(index=['CAF_Score'],
                           columns=[f"{mi}_{d}" for mi, d in zip(MI_OI_df["MI"], MI_OI_df["direction"])])

for MI_OI, direction in zip(MI_OI_df["MI"], MI_OI_df["direction"]):
    idx = MI_agg_meta.query("MI == @MI_OI and SR == @direction").index
    mi_vals = MI_SR_agg_cur_all[fibro_idx, :][:, idx].flatten()
    scatter_df = pd.DataFrame({
        "MI_Level": mi_vals,
        "CAF_Score": caf_score[fibro_idx]
    }).query("MI_Level >= 0.2")

    rho, p = spearmanr(scatter_df["MI_Level"], scatter_df["CAF_Score"])
    corr_CAF_df.loc["CAF_Score", f"{MI_OI}_{direction}"] = rho

plt.figure(figsize=(6, 4))
sns.heatmap(corr_CAF_df.astype(float), cmap="Reds", annot=True, fmt=".2f", vmin=0, vmax=1)
plt.title("Spearman Correlation: MI Sender vs CAF Score")
plt.savefig(file_savepath_main + "Corr_MI_Sender_CAF_Score.png", dpi=300, bbox_inches='tight')
plt.close()

# Step 4. Correlation with Tumor Functional State

In [12]:
geneset_df = pd.read_csv(data_path_main + "CancerSEA_OV/functional_geneset_list_df.csv")
geneset_df = geneset_df[geneset_df['Gene'].isin(adata_copy.var_names)]

geneset_dict = {
    term: np.where(np.isin(adata_copy.var_names,
                           geneset_df.loc[geneset_df['GeneSet'] == term, 'Gene']))[0]
    for term in np.unique(geneset_df['GeneSet'])
}

MI_OI_df = pd.DataFrame({
    "MI": ['MI10','MI13','MI6','MI9','MI12'],
    "direction": ['Receiver']*5
})
Malignant_COI = "Malignant_4"
malignant_idx = np.where(cellclass_updated == Malignant_COI)[0]
corr_tumor_state_df = pd.DataFrame(index=geneset_dict.keys(),
                                   columns=[f"{mi}_{d}" for mi, d in zip(MI_OI_df["MI"], MI_OI_df["direction"])])

for state, gene_idx in geneset_dict.items():
    exp_state = np.mean(exp_nor_all_norm[:, gene_idx], axis=1)
    exp_state_malignant = exp_state[malignant_idx]

    for MI_OI, direction in zip(MI_OI_df["MI"], MI_OI_df["direction"]):
        idx = MI_agg_meta.query("MI == @MI_OI and SR == @direction").index
        mi_vals = MI_SR_agg_cur_all[malignant_idx, :][:, idx].flatten()
        scatter_df = pd.DataFrame({
            "MI_Level": mi_vals,
            "State_Score": exp_state_malignant
        })

        rho, p = spearmanr(scatter_df["MI_Level"], scatter_df["State_Score"])
        corr_tumor_state_df.loc[state, f"{MI_OI}_{direction}"] = rho

plt.figure(figsize=(8, 6))
sns.heatmap(corr_tumor_state_df.astype(float), cmap="Reds", annot=True, fmt=".2f", vmin=0, vmax=1)
plt.title("Spearman Correlation: MI Receiver vs Tumor Functional States (Malignant_4)")
plt.savefig(file_savepath_main + "Corr_MI_Receiver_TumorFunctionalState_Malignant4.png",
            dpi=300, bbox_inches='tight')
plt.close()
